# 02 · Feature Engineering
**Project 18 — Decision Network for Intelligent Recommendation Systems**

We build the discretised user/item profiles that feed into the Bayesian Network nodes:

| BN Node | Source | States |
|---------|--------|--------|
| `user_activity` | Rating count per user | low / medium / high |
| `user_rating_bucket` | Mean rating per user | harsh / neutral / generous |
| `user_gender` | Demographic | M / F |
| `item_genre` | u.item metadata | 12 genres |
| `item_popularity` | Rating count per item | niche / moderate / popular |
| `item_quality` | Mean rating per item | low / medium / high |
| `rating_positive` | rating ≥ 4 | 0 / 1 |


In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from src.data_loader import load_all, train_test_split_temporal
from src.feature_engineering import (
    build_user_profiles, build_item_profiles,
    build_user_item_matrix, build_dn_training_data
)


plt.rcParams.update({
    'figure.facecolor': '#0f1525', 'axes.facecolor': '#0f1525',
    'axes.edgecolor': '#2d3748',   'axes.labelcolor': '#a0aec0',
    'xtick.color': '#718096',      'ytick.color': '#718096',
    'text.color': '#e2e8f0',       'grid.color': '#1a2035',
    'grid.linestyle': '--',
})
PALETTE = ['#8b5cf6', '#06b6d4', '#10b981', '#f59e0b', '#f43f5e']

ratings, users, items, merged = load_all()
train_r, test_r = train_test_split_temporal(ratings)
train_merged = merged[merged.index.isin(train_r.index)]
print('Data loaded ✓')

## 2.1  Build User & Item Profiles

In [ ]:
user_profiles = build_user_profiles(train_merged)
item_profiles = build_item_profiles(train_merged)

print('── User Profiles ─────────────────────────────────────')
print(user_profiles[['user_id','avg_rating','rating_count','activity','rating_bucket','fav_genre']].head(8).to_string(index=False))

print('\n── Item Profiles ─────────────────────────────────────')
print(item_profiles[['item_id','title','primary_genre','avg_rating','rating_count','popularity','quality']].head(8).to_string(index=False))

## 2.2  Visualise Discretisation Bins

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Feature Discretisation', fontsize=14, fontweight='bold', color='#e2e8f0')

# 1. User activity distribution
ax = axes[0][0]
activity_counts = user_profiles['activity'].value_counts().reindex(['low','medium','high'])
bars = ax.bar(activity_counts.index, activity_counts.values,
              color=[PALETTE[4], PALETTE[0], PALETTE[2]], edgecolor='none', alpha=0.85)
ax.set_title('User Activity Levels', color='#e2e8f0')
ax.set_ylabel('Number of Users'); ax.grid(axis='y', alpha=0.3)
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+1, str(int(b.get_height())),
            ha='center', fontsize=10, color='#a78bfa')

# 2. User rating bucket
ax = axes[0][1]
rb_counts = user_profiles['rating_bucket'].value_counts().reindex(['harsh','neutral','generous'])
bars = ax.bar(rb_counts.index, rb_counts.values,
              color=[PALETTE[4], PALETTE[1], PALETTE[2]], edgecolor='none', alpha=0.85)
ax.set_title('User Rating Style', color='#e2e8f0')
ax.set_ylabel('Number of Users'); ax.grid(axis='y', alpha=0.3)
for b in bars:
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+1, str(int(b.get_height())),
            ha='center', fontsize=10, color='#67e8f9')

# 3. Item quality distribution
ax = axes[1][0]
q_counts = item_profiles['quality'].value_counts().reindex(['low','medium','high'])
ax.bar(q_counts.index, q_counts.values,
       color=[PALETTE[4], PALETTE[0], PALETTE[2]], edgecolor='none', alpha=0.85)
ax.set_title('Item Quality Bins', color='#e2e8f0')
ax.set_ylabel('Number of Movies'); ax.grid(axis='y', alpha=0.3)

# 4. Item popularity distribution
ax = axes[1][1]
p_counts = item_profiles['popularity'].value_counts().reindex(['niche','moderate','popular'])
ax.bar(p_counts.index, p_counts.values,
       color=[PALETTE[4], PALETTE[1], PALETTE[2]], edgecolor='none', alpha=0.85)
ax.set_title('Item Popularity Bins', color='#e2e8f0')
ax.set_ylabel('Number of Movies'); ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/fig_discretisation.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.3  Build DN Training Data

In [ ]:
dn_data = build_dn_training_data(train_merged, user_profiles, item_profiles)
print(f'DN Training rows : {len(dn_data):,}')
print(f'Positive fraction: {dn_data["rating_positive"].astype(int).mean():.3f}')
print()
print(dn_data.head(8).to_string(index=False))

## 2.4  Conditional Heatmap — P(positive | quality, activity)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Empirical Conditional Probabilities P(like=1 | features)', fontsize=13, color='#e2e8f0')

# P(like | quality, activity)
ax = axes[0]
pivot1 = (
    dn_data.groupby(['item_quality', 'user_activity'])['rating_positive']
    .apply(lambda x: x.astype(int).mean())
    .unstack()
    .reindex(index=['low','medium','high'], columns=['low','medium','high'])
)
sns.heatmap(pivot1, ax=ax, annot=True, fmt='.2f', cmap='magma',
            linewidths=0.5, linecolor='#0f1525',
            cbar_kws={'label': 'P(like=1)'},
            annot_kws={'size': 11, 'color': 'white'})
ax.set_title('P(like | item quality, user activity)', color='#e2e8f0')
ax.set_xlabel('user_activity'); ax.set_ylabel('item_quality')

# P(like | rating_bucket, quality)
ax = axes[1]
pivot2 = (
    dn_data.groupby(['user_rating_bucket', 'item_quality'])['rating_positive']
    .apply(lambda x: x.astype(int).mean())
    .unstack()
    .reindex(index=['harsh','neutral','generous'], columns=['low','medium','high'])
)
sns.heatmap(pivot2, ax=ax, annot=True, fmt='.2f', cmap='viridis',
            linewidths=0.5, linecolor='#0f1525',
            cbar_kws={'label': 'P(like=1)'},
            annot_kws={'size': 11, 'color': 'white'})
ax.set_title('P(like | rating style, item quality)', color='#e2e8f0')
ax.set_xlabel('item_quality'); ax.set_ylabel('user_rating_bucket')

plt.tight_layout()
plt.savefig('../data/fig_conditional_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ Notebook 02 complete. Proceed to 03_decision_network_model.ipynb')